# Nearest neighbor

Equal-temperature reservoirs provide an equilibrium reference; unequal
reservoirs drive heat transport and density redistribution. Both runs
start independently from the same uniform distribution. This example uses
the built-in nearest-neighbor model and requires no calibration data.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from matplotlib import pyplot as plt

N_STEPS = 2000
REPORT_EVERY = 100
dt = 0.04  # Time per Markov step, as in the source demonstration.
EARLY_SAMPLES = min(120, N_STEPS // 2 + 1)
BATH_BETAS = {
    "Equilibrium": (1.5, 1.5),
    "Nonequilibrium": (1.1, 1.9),
}

## Nearest-neighbor dynamics and boundary conditions

Set the lattice size explicitly to 25 sites and 5 energy levels. Omitting
`connectivity` from `Model` selects the built-in nearest-neighbor graph:
each source site can move left, stay, or move right, with every source and
destination energy-level pair structurally allowed. The boundary walls
reflect outward moves back onto the boundary site. Consequently, staying
at either boundary has multiplicity two, while an ordinary allowed move
has multiplicity one.

Omitting `degeneracy` gives unit degeneracy at every site and level. The
default energies are $\epsilon_i=i^2/2$ for $i=1,\ldots,5$:
`[0.5, 2, 4.5, 8, 12.5]`. The model combines the graph weights and Boltzmann
factors to construct each normalized transition kernel.

Only the first and last sites are thermostatted. Their inverse
temperatures are $(1.5,1.5)$ or $(1.1,1.9)$; the interior inverse
temperatures are solved from local energy balance at every step.

In [ ]:
nsites = 25
nlevels = 5
dx = 1.0 / nsites
sites = np.arange(1, nsites + 1)

print(f"Nearest-neighbor lattice: {nsites} sites, {nlevels} energy levels")
print(f"Steps per case: {N_STEPS:,}; final time: {dt * N_STEPS:g}")

## Simulation and Kullback-Leibler diagnostic

Simulation start with $p_{\ell i}=1/(25\times5)$.

Both, equilibrium and nonequilibrium, cases run in separate spawned processes using the `npy` kernel.
The shared `_parallel_simulation.py` module keeps workers importable.

To compare the conditional energy distribution
$p(i\mid\ell)=p_{\ell i}/\sum_j p_{\ell j}$ with the local Boltzmann
distribution $q(i\mid\ell)\propto g_{\ell i}e^{-\beta_\ell\epsilon_i}$, we evaluate the Kullback-Leibler divergence

$$D_\ell=\sum_i p(i\mid\ell)
\log\frac{p(i\mid\ell)}{q(i\mid\ell)}.$$

For the nearest-neighbor dynamical graph, the structural weight depends only on the source and
destination sites, not on their energy levels. The target factor
$e^{-\beta_\ell\epsilon_i}$ can therefore be taken outside the sum over
source states. Each update produces the local Boltzmann conditional
distribution exactly in exact arithmetic, even while the temperature and
density profiles evolve. The computed KL values consequently measure
floating-point roundoff rather than a physical departure from local
equilibrium.

**NB!**: simulations may take some time...

In [ ]:
# Support launching from either the repository root or examples/.
helper_dir = next(
    path.resolve()
    for path in (Path("."), Path("examples"))
    if (path / "_parallel_simulation.py").is_file()
)
for path in (helper_dir.parent, helper_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _parallel_simulation import run_cases  # noqa: E402 (path setup above)

results = run_cases(
    BATH_BETAS,
    nsites,
    nlevels,
    N_STEPS,
    REPORT_EVERY,
)

## Temperature and density after 2,000 steps

Temperature is $T_\ell=1/\beta_\ell$ in reduced units, whereas density is evaluated as
$\rho_\ell=\sum_i p_{\ell i}/\Delta x$ and normalized so that $\sum_\ell\rho_\ell\Delta x=1$.

Density remains close to one in both cases. The density panels resolve
small deviations using shared limits and show the actual density values
without an axis offset.

In [ ]:
ORANGE = "#F97316"
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
    }
)


def style_axis(ax):
    ax.grid(True, color="#E2E8F0", linewidth=0.6)


fig_profiles, axes = plt.subplots(
    2, 2, figsize=(9, 6), sharex=True, sharey="col", layout="constrained"
)
for row, (name, result) in enumerate(results.items()):
    state = result["model"].state
    assert np.all(state.betas > 0)
    density = state.site_occupancies() / dx
    np.testing.assert_allclose(density.sum() * dx, 1.0, atol=1e-12, rtol=0)
    profiles = (1.0 / state.betas, density)
    for col, (ax, profile) in enumerate(zip(axes[row], profiles, strict=True)):
        ax.plot(
            sites,
            profile,
            color=ORANGE,
            marker="o",
            markerfacecolor="none",
            markersize=5,
            linestyle="none",
            label="Nearest neighbors",
        )
        quantity = "Temperature" if col == 0 else "Density"
        ax.set_title(f"{name} · {quantity}")
        ax.set_ylabel(r"$T_\ell$" if col == 0 else r"$\rho_\ell$")
        ax.set_xticks([1, 7, 13, 19, 25])
        ax.set_xlim(0.5, nsites + 0.5)
        ax.set_ylim(*((0.5, 1.0) if col == 0 else (0.8, 1.3)))
        if col == 1:
            ax.ticklabel_format(axis="y", style="plain", useOffset=False)
        style_axis(ax)
        if row == 1:
            ax.set_xlabel(r"Site, $\ell$")

fig_profiles.suptitle(f"Nearest neighbors · after {N_STEPS:,} steps")
plt.show()

## Nonequilibrium local-equilibrium KL diagnostic

The orange band spans the minimum and maximum KL divergence across all
25 sites, including the reservoirs; it is a **spatial range**, not an
uncertainty interval. The left panel shows the full trajectory and the
right shows the first 120 samples (steps 0–119). Time is $t=0.04\,n$ for
step $n$; only the vertical axis is logarithmic.

For these nearest-neighbor dynamics, KL is zero in exact arithmetic.
Values at this scale are numerical roundoff, not physical relaxation.

The macroscopic profiles can still evolve while their
conditional energy distributions remain locally Boltzmann.

In [ ]:
kl_ne = results["Nonequilibrium"]["kl"]
time = dt * np.arange(N_STEPS + 1)
lower = np.nanmin(kl_ne, axis=1)
upper = np.nanmax(kl_ne, axis=1)
plotted_lower = np.ma.masked_less_equal(lower, 0)
plotted_upper = np.ma.masked_less_equal(upper, 0)

fig_kl, axes_kl = plt.subplots(
    1, 2, figsize=(9, 3.4), sharey=True, layout="constrained"
)
windows = [slice(None), slice(0, EARLY_SAMPLES)]
titles = ["Full trajectory", f"First {EARLY_SAMPLES} samples"]
for ax, window, title in zip(axes_kl, windows, titles, strict=True):
    ax.fill_between(
        time[window],
        plotted_lower[window],
        plotted_upper[window],
        color=ORANGE,
        alpha=0.5,
        linewidth=0,
        label="Spatial min–max",
    )
    ax.plot(
        time[window],
        plotted_upper[window],
        color=ORANGE,
        linewidth=0.7,
        label="Site maximum",
    )
    ax.set_yscale("log")
    if np.nanmax(np.abs(kl_ne)) < 1e-12:
        ax.set_ylim(1e-18, 1e-14)
        ax.text(
            0.5,
            0.10,
            "KL is zero within floating-point precision",
            transform=ax.transAxes,
            ha="center",
            fontsize=8,
        )
    ax.set_xlim(time[window][0], time[window][-1])
    ax.set_title(title)
    ax.set_xlabel("Time (reduced units)")
    style_axis(ax)
axes_kl[0].set_ylabel("Local-equilibrium KL divergence")
axes_kl[0].legend(frameon=False)
fig_kl.suptitle("Nearest neighbors · nonequilibrium KL diagnostic")
plt.show()